# M6 — OpenMax + Weibull Baseline (Open-Set Unknown Detection)
**Owner:** Member B (Disease-Diagnosis & OWL Lead)  
**Requires:** M1 (Provisional CNN Backbone)  
**Purpose:** Reimplement the OpenMax/Weibull prior-method baseline (Bendale & Boult, CVPR 2016; Karim, Mahmud & Khan 2024) as the comparison target that M15 (cross-task consistency) must beat.

**Key Design Decisions (fixes from v1):**
- MAV computed on **M1's raw 256-dim embeddings** (not disease head internals) to preserve activation variance
- OpenMax revision applied to **softmax probabilities** (not raw logits) for meaningful probability redistribution
- MAV computed from **all** training samples (not only correctly-classified) for statistical robustness on small dataset
- Simplified disease head: **no LayerNorm** (prevents activation collapse)
- Adaptive Weibull tailsize with robust fallback for degenerate distributions

---

## Cell 1 — Dependencies

In [1]:
# ============================================================
# CELL 1 — DEPENDENCIES
# ============================================================
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Ensure required packages
for pkg_import, pkg_name in [("librosa", "librosa"), ("soundfile", "soundfile"),
                              ("sklearn", "scikit-learn"), ("scipy", "scipy"),
                              ("seaborn", "seaborn")]:
    try:
        __import__(pkg_import)
    except ImportError:
        print(f"Installing {pkg_name}...")
        pip_install(pkg_name)

print("Dependencies OK.")

Dependencies OK.


## Cell 2 — Global Configuration
All preprocessing parameters **must match M1 exactly** for the frozen backbone to produce valid embeddings.

In [2]:
# ============================================================
# CELL 2 — GLOBAL CONFIGURATION
# ============================================================
import os, math, glob

# ---- ICBHI DATASET PATHS (Kaggle) ----
_BASE = (
    "/kaggle/input/respiratory-sound-database"
    "/Respiratory_Sound_Database/Respiratory_Sound_Database"
)
DATA_ROOT  = _BASE + "/audio_and_txt_files"
SPLIT_FILE = _BASE + "/ICBHI_Challenge_train_test.txt"
DIAG_FILE  = _BASE + "/patient_diagnosis.csv"

# Fallback paths for different Kaggle dataset versions
if not os.path.exists(DATA_ROOT):
    _ALT_BASES = [
        "/kaggle/input/datasets/vbookshelf/respiratory-sound-database"
        "/Respiratory_Sound_Database/Respiratory_Sound_Database",
        "/kaggle/input/icbhi-respiratory-sound-database",
    ]
    for alt in _ALT_BASES:
        test_audio = alt + "/audio_and_txt_files"
        if os.path.exists(test_audio) or os.path.exists(alt):
            _BASE = alt
            DATA_ROOT  = test_audio if os.path.exists(test_audio) else alt
            SPLIT_FILE = _BASE + "/ICBHI_Challenge_train_test.txt"
            DIAG_FILE  = _BASE + "/patient_diagnosis.csv"
            break

# Search for DIAG_FILE recursively across Kaggle input if not found at default path
if not os.path.exists(DIAG_FILE):
    diag_hits = (
        glob.glob("/kaggle/input/**/patient_diagnosis.csv", recursive=True) +
        glob.glob("/kaggle/input/**/*diagnosis*.txt", recursive=True) +
        glob.glob("/kaggle/input/**/*diagnosis*.csv", recursive=True)
    )
    if diag_hits:
        DIAG_FILE = diag_hits[0]

# ---- M1 checkpoint path (add as Kaggle dataset input) ----
M1_CKPT_PATH = None
_m1_candidates = [
    "/kaggle/input/m1-provisional-cnn-backbone/best_model.pth",
    "/kaggle/input/m1-cnn-backbone/best_model.pth",
    "/kaggle/input/m1-backbone/best_model.pth",
]
for p in _m1_candidates:
    if os.path.exists(p):
        M1_CKPT_PATH = p
        break

if M1_CKPT_PATH is None:
    hits = glob.glob("/kaggle/input/**/best_model.pth", recursive=True)
    if hits:
        M1_CKPT_PATH = hits[0]

print(f"DATA_ROOT exists : {os.path.exists(DATA_ROOT)}")
print(f"DIAG_FILE        : {DIAG_FILE} (exists: {os.path.exists(DIAG_FILE)})")
print(f"M1_CKPT_PATH     : {M1_CKPT_PATH} (exists: {os.path.exists(M1_CKPT_PATH) if M1_CKPT_PATH else False})")

# ---- Preprocessing config (MUST match M1 exactly) ----
CFG = {
    # Audio (synced with M1)
    "sample_rate"    : 16000,
    "duration_s"     : 8.0,
    "n_mels"         : 128,
    "n_fft"          : 512,
    "hop_length"     : 160,
    "win_length"     : 400,
    "f_min"          : 50,
    "f_max"          : 2000,
    "n_samples"      : int(16000 * 8.0),
    "n_frames"       : None,

    # Training — Disease head on frozen M1 backbone
    "batch_size"     : 32,
    "num_epochs"     : 80,
    "lr"             : 5e-4,
    "weight_decay"   : 1e-4,
    "lr_step_size"   : 25,
    "lr_gamma"       : 0.5,
    "seed"           : 42,

    # Disease classes
    "known_diseases"   : ["COPD", "Healthy", "URTI"],
    "unknown_diseases" : ["Bronchiectasis", "Pneumonia", "Bronchiolitis"],
    "num_known_classes": 3,
    "sound_classes"    : ["Normal", "Crackle", "Wheeze", "Both"],
    "num_sound_classes": 4,

    # OpenMax hyperparameters
    "openmax_tailsize" : 20,
    "openmax_alpha"    : 2,
    "openmax_threshold": 0.5,

    # Paths
    "data_root"   : DATA_ROOT,
    "ckpt_dir"    : "/kaggle/working/checkpoints_M6",
    "results_dir" : "/kaggle/working/results_M6",
}

CFG["n_frames"] = 1 + math.floor(CFG["n_samples"] / CFG["hop_length"])

os.makedirs(CFG["ckpt_dir"], exist_ok=True)
os.makedirs(CFG["results_dir"], exist_ok=True)

disease_to_idx = {d: i for i, d in enumerate(CFG["known_diseases"])}
idx_to_disease = {i: d for d, i in disease_to_idx.items()}

print("\nConfiguration:")
for k, v in CFG.items():
    print(f"  {k:<20} = {v}")
print(f"\nDisease mapping: {disease_to_idx}")


DATA_ROOT exists : True
DIAG_FILE        : /kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/patient_diagnosis.csv (exists: True)
M1_CKPT_PATH     : /kaggle/input/datasets/sudamchandrabasak/best-model/best_model.pth (exists: True)

Configuration:
  sample_rate          = 16000
  duration_s           = 8.0
  n_mels               = 128
  n_fft                = 512
  hop_length           = 160
  win_length           = 400
  f_min                = 50
  f_max                = 2000
  n_samples            = 128000
  n_frames             = 801
  batch_size           = 32
  num_epochs           = 80
  lr                   = 0.0005
  weight_decay         = 0.0001
  lr_step_size         = 25
  lr_gamma             = 0.5
  seed                 = 42
  known_diseases       = ['COPD', 'Healthy', 'URTI']
  unknown_diseases     = ['Bronchiectasis', 'Pneumonia', 'Bronchiolitis']
  num_known_classes    = 3
  sound_classes        = ['Normal'

## Cell 3 — Imports & Reproducibility

In [3]:
# ============================================================
# CELL 3 — IMPORTS & REPRODUCIBILITY
# ============================================================
import os, json, math, time, glob, random, datetime, tempfile, sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import librosa

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torch.optim as optim

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve,
)
from sklearn.model_selection import GroupShuffleSplit
from collections import Counter
import scipy.stats as stats

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# ---- Reproducibility ----
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CFG["seed"])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU   : {torch.cuda.get_device_name(0)}")

Device: cuda
GPU   : Tesla T4


## Cell 4 — M1 CNN Backbone (Frozen Encoder)
Exact copy of M1's architecture. The encoder is **frozen** — only the disease head trains.
The `get_embedding()` method returns the 256-dim features used for both the disease head AND for OpenMax MAV computation.

In [4]:
# ============================================================
# CELL 4 — M1 CNN BACKBONE (FROZEN)
# ============================================================

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool_kernel=(2, 2)):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=pool_kernel),
        )
    def forward(self, x):
        return self.block(x)


class M1_CNN(nn.Module):
    """
    M1 Provisional CNN Backbone.
    Input  : (B, 1, n_mels, n_frames)
    Output : (B, 4) logits for sound-event classification
    """
    def __init__(self, num_classes=4, dropout=0.5):
        super().__init__()
        self.encoder = nn.Sequential(
            ConvBlock(1,   32),
            ConvBlock(32,  64),
            ConvBlock(64,  128),
            ConvBlock(128, 256),
        )
        self.gap     = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout)
        self.head    = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        feat = self.encoder(x)
        feat = self.gap(feat)
        feat = feat.view(feat.size(0), -1)
        feat = self.dropout(feat)
        return self.head(feat)

    def get_embedding(self, x):
        """Returns 256-dim embedding (before dropout and classification head)."""
        feat = self.encoder(x)
        feat = self.gap(feat)
        return feat.view(feat.size(0), -1)


# ---- Load M1 checkpoint ----
m1_model = M1_CNN(num_classes=CFG["num_sound_classes"]).to(DEVICE)

if M1_CKPT_PATH and os.path.exists(M1_CKPT_PATH):
    state = torch.load(M1_CKPT_PATH, map_location=DEVICE, weights_only=False)
    if "model_state" in state:
        m1_model.load_state_dict(state["model_state"])
        print(f"Loaded M1 checkpoint from epoch {state.get('epoch', '?')}")
    elif "state_dict" in state:
        m1_model.load_state_dict(state["state_dict"])
        print("Loaded M1 checkpoint (state_dict format)")
    else:
        m1_model.load_state_dict(state)
        print("Loaded M1 checkpoint (raw state_dict)")
else:
    print("ERROR: M1 checkpoint not found!")
    print("Add M1's output as a Kaggle dataset input before running.")
    raise FileNotFoundError("M1 checkpoint required — add it as Kaggle dataset input.")

# Freeze everything
m1_model.eval()
for p in m1_model.parameters():
    p.requires_grad = False

n_params = sum(p.numel() for p in m1_model.parameters())
print(f"M1 encoder: {n_params:,} parameters (all frozen)")

# Quick embedding check
with torch.no_grad():
    dummy = torch.randn(2, 1, CFG["n_mels"], CFG["n_frames"]).to(DEVICE)
    emb = m1_model.get_embedding(dummy)
    print(f"Embedding shape: {emb.shape}  (expected: (2, 256))")
    print(f"Embedding stats: mean={emb.mean():.4f}, std={emb.std():.4f}, "
          f"min={emb.min():.4f}, max={emb.max():.4f}")
    print(f"Non-zero ratio: {(emb != 0).float().mean():.4f}")

Loaded M1 checkpoint from epoch 29
M1 encoder: 421,732 parameters (all frozen)
Embedding shape: torch.Size([2, 256])  (expected: (2, 256))
Embedding stats: mean=9.0443, std=13.5017, min=0.0000, max=60.3510
Non-zero ratio: 0.7285


## Cell 5 — ICBHI Dataset Parser + Disease Labels
Parses ICBHI 2017 cycle annotations and loads `patient_diagnosis.csv` for disease label mapping.

In [5]:
# ============================================================
# CELL 5 — ICBHI 2017 DATASET PARSER + DISEASE LABELS
# ============================================================
import glob

def parse_annotation_file(txt_path):
    """Parse ICBHI cycle annotation: start end crackle wheeze."""
    cycles = []
    with open(txt_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 4:
                continue
            try:
                start   = float(parts[0])
                end     = float(parts[1])
                crackle = int(parts[2])
                wheeze  = int(parts[3])
            except ValueError:
                continue
            if crackle == 0 and wheeze == 0:
                label = 0
            elif crackle == 1 and wheeze == 0:
                label = 1
            elif crackle == 0 and wheeze == 1:
                label = 2
            else:
                label = 3
            cycles.append({"start": start, "end": end,
                           "crackle": crackle, "wheeze": wheeze, "label": label})
    return cycles


def patient_id_from_stem(stem):
    """Extract numeric patient ID: '101_1b1_Al_sc_Meditron' -> 101."""
    try:
        return int(stem.split("_")[0])
    except (ValueError, IndexError):
        return -1


def load_diagnosis_map(data_root):
    """Load patient_id -> diagnosis from ICBHI's patient_diagnosis.csv or ICBHI_Challenge_diagnosis.txt."""
    candidates = [
        DIAG_FILE,
        os.path.join(os.path.dirname(data_root), "patient_diagnosis.csv"),
        os.path.join(data_root, "patient_diagnosis.csv"),
        os.path.join(os.path.dirname(data_root), "ICBHI_Challenge_diagnosis.txt"),
        os.path.join(data_root, "ICBHI_Challenge_diagnosis.txt"),
    ]
    extra_candidates = (
        glob.glob("/kaggle/input/**/patient_diagnosis.csv", recursive=True) +
        glob.glob("/kaggle/input/**/*diagnosis*.txt", recursive=True) +
        glob.glob("/kaggle/input/**/*diagnosis*.csv", recursive=True)
    )
    for c in extra_candidates:
        if c not in candidates:
            candidates.append(c)

    for path in candidates:
        if path and os.path.exists(path):
            diag_map = {}
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                for line in f:
                    # Handle CSV (comma), TSV (tab), or space-separated lines
                    clean_line = line.replace(",", " ").replace("\t", " ").strip()
                    parts = clean_line.split()
                    if len(parts) >= 2:
                        try:
                            pid  = int(parts[0])
                            diag = parts[1].strip()
                            diag_map[pid] = diag
                        except ValueError:
                            continue
            if diag_map:
                print(f"Loaded diagnosis map from: {path} ({len(diag_map)} patients)")
                return diag_map
    print("WARNING: Diagnosis file not found!")
    return {}


def build_cycle_dataframe(data_root):
    """Build cycle-level DataFrame with disease diagnosis labels."""
    sub = os.path.join(data_root, "ICBHI_final_database")
    audio_dir = sub if os.path.isdir(sub) else data_root

    wav_files = glob.glob(os.path.join(audio_dir, "*.wav"))
    txt_files = glob.glob(os.path.join(audio_dir, "*.txt"))
    print(f"Found {len(wav_files)} .wav files, {len(txt_files)} .txt files")

    diag_map = load_diagnosis_map(data_root)

    wav_stems = set(os.path.splitext(os.path.basename(w))[0] for w in wav_files)

    rows = []
    for txt_path in sorted(txt_files):
        stem = os.path.splitext(os.path.basename(txt_path))[0]
        if stem not in wav_stems:
            continue
        wav_path = os.path.join(audio_dir, stem + ".wav")
        if not os.path.exists(wav_path):
            continue

        pid = patient_id_from_stem(stem)
        diagnosis = diag_map.get(pid, "Unknown")

        for c in parse_annotation_file(txt_path):
            rows.append({
                "wav_path"   : wav_path,
                "stem"       : stem,
                "patient_id" : pid,
                "start"      : c["start"],
                "end"        : c["end"],
                "crackle"    : c["crackle"],
                "wheeze"     : c["wheeze"],
                "sound_label": c["label"],
                "diagnosis"  : diagnosis,
            })

    df = pd.DataFrame(rows)
    print(f"\nTotal cycles: {len(df)}")
    print(f"\nDiagnosis distribution:")
    for diag, grp in df.groupby("diagnosis"):
        n_patients = grp["patient_id"].nunique()
        print(f"  {diag:20s}: {n_patients:3d} patients, {len(grp):5d} cycles")
    return df


# ---- Build DataFrame ----
df_all = build_cycle_dataframe(CFG["data_root"])

# ---- Known / Unknown split ----
df_known   = df_all[df_all["diagnosis"].isin(CFG["known_diseases"])].copy()
df_unknown = df_all[df_all["diagnosis"].isin(CFG["unknown_diseases"])].copy()

# Assign disease label indices
df_known["disease_label"] = df_known["diagnosis"].map(disease_to_idx)
df_unknown["disease_label"] = -1

df_known   = df_known.reset_index(drop=True)
df_unknown = df_unknown.reset_index(drop=True)

print(f"\nKnown  : {df_known['patient_id'].nunique()} patients, {len(df_known)} cycles")
print(f"Unknown: {df_unknown['patient_id'].nunique()} patients, {len(df_unknown)} cycles")
print(f"\nKnown class distribution:")
for d in CFG["known_diseases"]:
    mask = df_known["diagnosis"] == d
    print(f"  {d:15s}: {df_known.loc[mask, 'patient_id'].nunique():3d} patients, "
          f"{mask.sum():5d} cycles, label={disease_to_idx[d]}")


Found 920 .wav files, 920 .txt files
Loaded diagnosis map from: /kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/patient_diagnosis.csv (126 patients)

Total cycles: 6898

Diagnosis distribution:
  Asthma              :   1 patients,     6 cycles
  Bronchiectasis      :   7 patients,   104 cycles
  Bronchiolitis       :   6 patients,   160 cycles
  COPD                :  64 patients,  5746 cycles
  Healthy             :  26 patients,   322 cycles
  LRTI                :   2 patients,    32 cycles
  Pneumonia           :   6 patients,   285 cycles
  URTI                :  14 patients,   243 cycles

Known  : 104 patients, 6311 cycles
Unknown: 19 patients, 549 cycles

Known class distribution:
  COPD           :  64 patients,  5746 cycles, label=0
  Healthy        :  26 patients,   322 cycles, label=1
  URTI           :  14 patients,   243 cycles, label=2


## Cell 6 — Patient-Independent Train/Test Split (Known Classes)
~70% patients for training, ~30% for testing. All unknowns reserved for open-set eval.

In [6]:
# ============================================================
# CELL 6 — PATIENT-INDEPENDENT TRAIN/TEST SPLIT
# ============================================================

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=CFG["seed"])
train_idx, test_idx = next(gss.split(
    df_known, df_known["disease_label"], groups=df_known["patient_id"]
))

df_known_train = df_known.iloc[train_idx].reset_index(drop=True)
df_known_test  = df_known.iloc[test_idx].reset_index(drop=True)

# Verify patient independence
train_pids = set(df_known_train["patient_id"].unique())
test_pids  = set(df_known_test["patient_id"].unique())
overlap = train_pids & test_pids
assert len(overlap) == 0, f"Patient leak! Overlapping: {overlap}"

print(f"Train: {len(train_pids)} patients, {len(df_known_train)} cycles")
print(f"Test : {len(test_pids)} patients, {len(df_known_test)} cycles")
print(f"Patient overlap: {len(overlap)} (must be 0)")

for split_name, split_df in [("Train", df_known_train), ("Test", df_known_test)]:
    print(f"\n{split_name} disease distribution:")
    for d in CFG["known_diseases"]:
        mask = split_df["diagnosis"] == d
        n_pts = split_df.loc[mask, "patient_id"].nunique()
        print(f"  {d:15s}: {n_pts:3d} patients, {mask.sum():5d} cycles")

Train: 72 patients, 4680 cycles
Test : 32 patients, 1631 cycles
Patient overlap: 0 (must be 0)

Train disease distribution:
  COPD           :  44 patients,  4279 cycles
  Healthy        :  20 patients,   256 cycles
  URTI           :   8 patients,   145 cycles

Test disease distribution:
  COPD           :  20 patients,  1467 cycles
  Healthy        :   6 patients,    66 cycles
  URTI           :   6 patients,    98 cycles


## Cell 7 — Log-Mel Spectrogram Dataset & DataLoaders
Preprocessing **exactly matches M1**: repeat-padding, [0, 1] normalization, same FFT params.

In [7]:
# ============================================================
# CELL 7 — DATASET & DATALOADERS
# ============================================================

def extract_log_mel(wav_path, start, end, cfg):
    """
    Extract log-mel spectrogram for one cycle.
    EXACTLY MATCHES M1: repeat-padding, [0,1] normalization, same FFT params.
    """
    sr = cfg["sample_rate"]
    target_len = cfg["n_samples"]

    try:
        y, orig_sr = librosa.load(wav_path, sr=None, offset=start, duration=end - start)
    except Exception:
        y = np.zeros(target_len, dtype=np.float32)
        orig_sr = sr

    if orig_sr != sr:
        y = librosa.resample(y, orig_sr=orig_sr, target_sr=sr)

    # Repeat-padding (matches M1)
    if len(y) < target_len:
        if len(y) > 0:
            reps = math.ceil(target_len / len(y))
            y = np.tile(y, reps)[:target_len]
        else:
            y = np.zeros(target_len, dtype=np.float32)
    else:
        y = y[:target_len]

    mel = librosa.feature.melspectrogram(
        y=y, sr=sr,
        n_fft=cfg["n_fft"],
        hop_length=cfg["hop_length"],
        win_length=cfg["win_length"],
        n_mels=cfg["n_mels"],
        fmin=cfg["f_min"],
        fmax=cfg["f_max"],
        power=2.0,
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)

    # [0, 1] normalization (matches M1)
    log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-8)

    # Ensure exact time dimension
    expected = cfg["n_frames"]
    if log_mel.shape[1] < expected:
        log_mel = np.pad(log_mel, ((0, 0), (0, expected - log_mel.shape[1])), mode="constant")
    else:
        log_mel = log_mel[:, :expected]

    return log_mel[np.newaxis, :, :].astype(np.float32)


class ICBHIDiseaseDataset(Dataset):
    def __init__(self, df, cfg, label_col="disease_label"):
        self.df = df.reset_index(drop=True)
        self.cfg = cfg
        self.label_col = label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        spec = extract_log_mel(row["wav_path"], row["start"], row["end"], self.cfg)
        spec_tensor = torch.from_numpy(spec)
        label = int(row[self.label_col]) if self.label_col in row.index else -1
        return spec_tensor, label, row["patient_id"]


# Create datasets
train_ds   = ICBHIDiseaseDataset(df_known_train, CFG)
test_ds    = ICBHIDiseaseDataset(df_known_test, CFG)
unknown_ds = ICBHIDiseaseDataset(df_unknown, CFG)

# Weighted sampler for class balance in training
train_labels_list = [df_known_train.iloc[i]["disease_label"] for i in range(len(train_ds))]
class_counts = Counter(train_labels_list)
sample_weights = [1.0 / class_counts[label] for label in train_labels_list]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], sampler=sampler,
                           num_workers=0, pin_memory=torch.cuda.is_available())
test_loader  = DataLoader(test_ds, batch_size=CFG["batch_size"], shuffle=False,
                           num_workers=0, pin_memory=torch.cuda.is_available())
unknown_loader = DataLoader(unknown_ds, batch_size=CFG["batch_size"], shuffle=False,
                             num_workers=0, pin_memory=torch.cuda.is_available())

print(f"Train   : {len(train_ds)} samples, {len(train_loader)} batches")
print(f"Test    : {len(test_ds)} samples, {len(test_loader)} batches")
print(f"Unknown : {len(unknown_ds)} samples, {len(unknown_loader)} batches")

# Quick check
spec, label, pid = train_ds[0]
print(f"\nSample spec shape: {spec.shape}  (expected: (1, 128, {CFG['n_frames']}))")
print(f"Sample range: [{spec.min():.3f}, {spec.max():.3f}] (expected: [0, 1])")

Train   : 4680 samples, 147 batches
Test    : 1631 samples, 51 batches
Unknown : 549 samples, 18 batches

Sample spec shape: torch.Size([1, 128, 801])  (expected: (1, 128, 801))
Sample range: [0.000, 1.000] (expected: [0, 1])


## Cell 8 — Disease Diagnosis Head
Simplified architecture: **no LayerNorm** (prevents activation collapse that caused 0.00 results in v1).
Returns both logits and penultimate-layer activations (needed for OpenMax).

In [8]:
# ============================================================
# CELL 8 — DISEASE DIAGNOSIS HEAD (SIMPLIFIED)
# ============================================================
#
# KEY FIX: No LayerNorm. The old version used LayerNorm -> GELU which
# collapsed activation variance, making all MAV distances near-zero
# and producing degenerate Weibull fits (=> 0.00 everywhere).
#
# Architecture: Linear(256->128) -> ReLU -> Dropout -> Linear(128->3)

class DiseaseHead(nn.Module):
    """
    Disease diagnosis head on frozen M1 embeddings.
    Returns (logits, penultimate_activations).
    """
    def __init__(self, embed_dim=256, hidden_dim=128, num_classes=3, dropout=0.3):
        super().__init__()
        self.fc1     = nn.Linear(embed_dim, hidden_dim)
        self.act     = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2     = nn.Linear(hidden_dim, num_classes)

    def forward(self, embedding):
        """Returns (logits, penultimate_activations)."""
        h = self.fc1(embedding)
        h = self.act(h)
        penultimate = h  # 128-dim — used for OpenMax
        h = self.dropout(h)
        logits = self.fc2(h)
        return logits, penultimate


disease_head = DiseaseHead(
    embed_dim=256,
    hidden_dim=128,
    num_classes=CFG["num_known_classes"],
    dropout=0.3,
).to(DEVICE)

n_trainable = sum(p.numel() for p in disease_head.parameters() if p.requires_grad)
n_total     = sum(p.numel() for p in disease_head.parameters())
print(f"Disease Head — Trainable: {n_trainable:,} | Total: {n_total:,}")

# Shape check
with torch.no_grad():
    dummy = torch.zeros(2, 256).to(DEVICE)
    logits, penult = disease_head(dummy)
    print(f"Logits shape: {logits.shape}  (expected: (2, 3))")
    print(f"Penultimate shape: {penult.shape}  (expected: (2, 128))")

Disease Head — Trainable: 33,283 | Total: 33,283
Logits shape: torch.Size([2, 3])  (expected: (2, 3))
Penultimate shape: torch.Size([2, 128])  (expected: (2, 128))


## Cell 9 — Checkpoint & Utility Functions

In [9]:
# ============================================================
# CELL 9 — CHECKPOINT UTILITIES
# ============================================================

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):   return int(obj)
        if isinstance(obj, np.floating):  return float(obj)
        if isinstance(obj, np.bool_):     return bool(obj)
        if isinstance(obj, np.ndarray):   return obj.tolist()
        return super().default(obj)


def save_checkpoint(model, optimizer, scheduler, epoch, metrics, path, is_best=False):
    state = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict() if scheduler else None,
        "metrics": metrics,
    }
    torch.save(state, path)
    if is_best:
        best_path = os.path.join(os.path.dirname(path), "best_model.pth")
        torch.save(state, best_path)


def load_checkpoint(model, optimizer, scheduler, ckpt_dir):
    latest = os.path.join(ckpt_dir, "latest.pth")
    if os.path.exists(latest):
        state = torch.load(latest, map_location=DEVICE, weights_only=False)
        model.load_state_dict(state["model_state"])
        optimizer.load_state_dict(state["optimizer_state"])
        if scheduler and state.get("scheduler_state"):
            scheduler.load_state_dict(state["scheduler_state"])
        start_epoch = state["epoch"] + 1
        print(f"Resumed from epoch {state['epoch']} (starting at {start_epoch})")
        return start_epoch
    return 0


def get_model_size_mb(model):
    with tempfile.NamedTemporaryFile(delete=True) as tmp:
        torch.save(model.state_dict(), tmp.name)
        return round(os.path.getsize(tmp.name) / (1024 * 1024), 2)


print("Utilities loaded.")

Utilities loaded.


## Cell 10 — Training Loop (Disease Head Only)
Trains the disease head on M1's frozen 256-dim embeddings. Only the head's ~33K parameters are updated.

In [10]:
# ============================================================
# CELL 10 — TRAINING LOOP
# ============================================================

# Class weights for imbalanced known classes
train_class_counts = Counter(df_known_train["disease_label"].tolist())
total_train = sum(train_class_counts.values())
class_weights = torch.tensor(
    [total_train / (CFG["num_known_classes"] * train_class_counts.get(i, 1))
     for i in range(CFG["num_known_classes"])],
    dtype=torch.float32
).to(DEVICE)
print(f"Class weights: {class_weights.cpu().tolist()}")

criterion  = nn.CrossEntropyLoss(weight=class_weights)
optimizer  = optim.Adam(disease_head.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler  = optim.lr_scheduler.StepLR(optimizer, step_size=CFG["lr_step_size"], gamma=CFG["lr_gamma"])

# Auto-resume
start_epoch = load_checkpoint(disease_head, optimizer, scheduler, CFG["ckpt_dir"])

history = {"train": []}
best_f1 = 0.0
best_epoch = -1
total_training_time = 0.0

print(f"\nTraining disease head for {CFG['num_epochs']} epochs...")
print(f"{'='*80}")

for epoch in range(start_epoch, CFG["num_epochs"]):
    epoch_start = time.time()

    # ---- TRAIN ----
    disease_head.train()
    train_loss = 0.0
    train_preds, train_labels = [], []

    for batch_specs, batch_labels, _ in train_loader:
        batch_specs  = batch_specs.to(DEVICE)
        batch_labels = batch_labels.to(DEVICE)

        with torch.no_grad():
            embeddings = m1_model.get_embedding(batch_specs)

        logits, _ = disease_head(embeddings)
        loss = criterion(logits, batch_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_specs.size(0)
        train_preds.extend(logits.argmax(dim=1).cpu().numpy())
        train_labels.extend(batch_labels.cpu().numpy())

    train_loss /= len(train_ds)
    train_acc = accuracy_score(train_labels, train_preds)
    train_f1  = f1_score(train_labels, train_preds, average="macro", zero_division=0)

    # ---- VALIDATE ----
    disease_head.eval()
    val_loss = 0.0
    val_preds, val_labels = [], []

    with torch.no_grad():
        for batch_specs, batch_labels, _ in test_loader:
            batch_specs  = batch_specs.to(DEVICE)
            batch_labels = batch_labels.to(DEVICE)

            embeddings = m1_model.get_embedding(batch_specs)
            logits, _ = disease_head(embeddings)
            loss = criterion(logits, batch_labels)

            val_loss += loss.item() * batch_specs.size(0)
            val_preds.extend(logits.argmax(dim=1).cpu().numpy())
            val_labels.extend(batch_labels.cpu().numpy())

    val_loss /= len(test_ds)
    val_acc = accuracy_score(val_labels, val_preds)
    val_f1  = f1_score(val_labels, val_preds, average="macro", zero_division=0)

    epoch_time = time.time() - epoch_start
    total_training_time += epoch_time

    is_best = val_f1 > best_f1
    if is_best:
        best_f1 = val_f1
        best_epoch = epoch

    history["train"].append({
        "epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
        "train_accuracy": train_acc, "val_accuracy": val_acc,
        "train_f1_macro": train_f1, "val_f1_macro": val_f1,
        "lr": optimizer.param_groups[0]["lr"], "epoch_time_s": epoch_time,
    })

    if epoch % 5 == 0 or is_best or epoch < 5:
        star = " *BEST*" if is_best else ""
        print(f"Epoch {epoch:3d}/{CFG['num_epochs']-1} | "
              f"TrLoss: {train_loss:.4f} TrAcc: {train_acc:.4f} TrF1: {train_f1:.4f} | "
              f"VaLoss: {val_loss:.4f} VaAcc: {val_acc:.4f} VaF1: {val_f1:.4f} | "
              f"{epoch_time:.1f}s{star}")

    # Save checkpoint
    metrics_snap = {"val_f1": val_f1, "val_acc": val_acc, "val_loss": val_loss}
    save_checkpoint(disease_head, optimizer, scheduler, epoch, metrics_snap,
                    os.path.join(CFG["ckpt_dir"], "latest.pth"), is_best=is_best)

    scheduler.step()

avg_epoch_time = total_training_time / max(CFG["num_epochs"] - start_epoch, 1)
print(f"\n{'='*80}")
print(f"Training complete. Best val F1: {best_f1:.4f} at epoch {best_epoch}")
print(f"Total time: {total_training_time:.1f}s ({avg_epoch_time:.1f}s/epoch)")

Class weights: [0.36457115411758423, 6.09375, 10.758620262145996]

Training disease head for 80 epochs...
Epoch   0/79 | TrLoss: 0.7161 TrAcc: 0.3581 TrF1: 0.2408 | VaLoss: 2.2912 VaAcc: 0.0674 VaF1: 0.0720 | 151.7s *BEST*
Epoch   1/79 | TrLoss: 0.6281 TrAcc: 0.4380 TrF1: 0.3762 | VaLoss: 1.8652 VaAcc: 0.2066 VaF1: 0.1677 | 124.2s *BEST*
Epoch   2/79 | TrLoss: 0.6008 TrAcc: 0.5423 TrF1: 0.5305 | VaLoss: 1.6599 VaAcc: 0.3636 VaF1: 0.2561 | 120.1s *BEST*
Epoch   3/79 | TrLoss: 0.5595 TrAcc: 0.5994 TrF1: 0.5923 | VaLoss: 1.6181 VaAcc: 0.4292 VaF1: 0.2864 | 119.0s *BEST*
Epoch   4/79 | TrLoss: 0.5614 TrAcc: 0.6434 TrF1: 0.6452 | VaLoss: 1.6562 VaAcc: 0.4476 VaF1: 0.2959 | 116.1s *BEST*
Epoch   5/79 | TrLoss: 0.5145 TrAcc: 0.6803 TrF1: 0.6804 | VaLoss: 1.5436 VaAcc: 0.4795 VaF1: 0.3094 | 115.5s *BEST*
Epoch   6/79 | TrLoss: 0.5077 TrAcc: 0.6840 TrF1: 0.6823 | VaLoss: 1.4195 VaAcc: 0.5273 VaF1: 0.3333 | 115.9s *BEST*
Epoch   8/79 | TrLoss: 0.4590 TrAcc: 0.7348 TrF1: 0.7359 | VaLoss: 1.2522 V

## Cell 11 — Feature Extraction Helper
Extracts M1 embeddings (256-dim), disease head activations (128-dim), logits, predictions from any dataloader.
**Both** embedding types are extracted — M1 embeddings are used for MAV, activations for diagnostics.

In [11]:
# ============================================================
# CELL 11 — FEATURE EXTRACTION
# ============================================================

@torch.no_grad()
def extract_features(m1, head, loader, device):
    """
    Extract all features needed for OpenMax.
    Returns:
        m1_embeddings:  (N, 256) — M1's raw embeddings (for MAV computation)
        activations:    (N, 128) — disease head penultimate layer
        logits:         (N, 3)   — disease head output logits
        predictions:    (N,)     — argmax predictions
        labels:         (N,)     — ground truth labels
        patient_ids:    (N,)     — patient IDs
    """
    m1.eval()
    head.eval()

    all_embs, all_acts, all_logits = [], [], []
    all_labels, all_pids = [], []

    for specs, labels, pids in loader:
        specs = specs.to(device)
        embs = m1.get_embedding(specs)         # (B, 256)
        logits, penult = head(embs)             # (B, 3), (B, 128)

        all_embs.append(embs.cpu().numpy())
        all_acts.append(penult.cpu().numpy())
        all_logits.append(logits.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_pids.extend([int(p) for p in pids])

    return (
        np.concatenate(all_embs),
        np.concatenate(all_acts),
        np.concatenate(all_logits),
        np.argmax(np.concatenate(all_logits), axis=1),
        np.array(all_labels),
        np.array(all_pids),
    )

print("Feature extraction function ready.")

Feature extraction function ready.


## Cell 12 — OpenMax + Weibull Core Implementation
Reference: Bendale & Boult, "Towards Open Set Deep Networks", CVPR 2016.

**Key differences from the broken v1:**
1. MAV is computed on **M1's 256-dim embeddings** (not disease head internals)
2. OpenMax revises **softmax probabilities** (not raw logits)
3. MAV uses **all** training samples per class (not just correctly classified)
4. Adaptive tailsize with robust Weibull fallback

In [12]:
# ============================================================
# CELL 12 — OPENMAX + WEIBULL IMPLEMENTATION (FIXED)
# ============================================================

def compute_mav_and_distances(embeddings, labels, num_classes, use_all=True, predictions=None):
    """
    Compute Mean Activation Vectors (MAVs) and distances.

    KEY FIX: uses ALL training samples per class (not just correctly classified)
    for statistical robustness on the small ICBHI dataset.

    Args:
        embeddings:  (N, D) — M1's 256-dim embeddings
        labels:      (N,)   — ground truth class labels
        num_classes: int
        use_all:     if True, use all samples; if False, only correctly classified
        predictions: (N,) — required if use_all=False

    Returns:
        mavs:      dict {class_id: (D,) array}
        distances: dict {class_id: (n_samples,) array}
    """
    mavs = {}
    distances = {}

    for c in range(num_classes):
        if use_all:
            mask = (labels == c)
        else:
            assert predictions is not None
            mask = (labels == c) & (predictions == c)

        class_embs = embeddings[mask]

        if len(class_embs) == 0:
            print(f"  WARNING: No samples for class {c}!")
            mavs[c] = np.zeros(embeddings.shape[1])
            distances[c] = np.array([1.0])
            continue

        mav = class_embs.mean(axis=0)
        mavs[c] = mav

        dists = np.linalg.norm(class_embs - mav, axis=1)
        distances[c] = dists

        print(f"  Class {c} ({idx_to_disease.get(c, '?')}): "
              f"{len(class_embs)} samples, "
              f"dist mean={dists.mean():.4f}, std={dists.std():.4f}, "
              f"min={dists.min():.4f}, max={dists.max():.4f}")

    return mavs, distances


def fit_weibull(distances, tailsize=20):
    """
    Fit Weibull distribution to the tail of distance distribution per class.
    Uses adaptive tailsize and robust fallback.
    """
    weibull_models = {}

    for c, dists in distances.items():
        # Adaptive tailsize: use at least 10, at most len(dists)//2
        effective_tail = max(10, min(tailsize, len(dists) // 2, len(dists)))

        sorted_dists = np.sort(dists)[::-1]  # descending
        tail = sorted_dists[:effective_tail]

        # Filter out zero/near-zero distances
        tail = tail[tail > 1e-8]

        if len(tail) < 3:
            fallback_scale = max(dists.max() * 2.0, 1.0)
            print(f"  Class {c}: too few valid tail distances ({len(tail)}), "
                  f"fallback scale={fallback_scale:.4f}")
            weibull_models[c] = (1.0, 0.0, fallback_scale)
            continue

        try:
            shape, loc, scale = stats.weibull_min.fit(tail, floc=0)

            # Safeguard: if scale is degenerate, use a robust fallback
            if scale <= 1e-6 or not np.isfinite(scale):
                scale = np.percentile(dists[dists > 0], 95) if (dists > 0).any() else 1.0
                scale = max(scale, 1e-3)
                print(f"  Class {c}: scale was degenerate, using p95 fallback={scale:.4f}")

            if not np.isfinite(shape) or shape <= 0:
                shape = 1.0
                print(f"  Class {c}: shape was invalid, using fallback=1.0")

            weibull_models[c] = (shape, loc, scale)
            print(f"  Class {c}: Weibull fit — shape={shape:.4f}, scale={scale:.4f} "
                  f"(tailsize={effective_tail})")

        except Exception as e:
            fallback_scale = max(dists.max() * 2.0, 1.0)
            print(f"  Class {c}: Weibull fit failed ({e}), fallback scale={fallback_scale:.4f}")
            weibull_models[c] = (1.0, 0.0, fallback_scale)

    return weibull_models


def openmax_recalibrate(logits_single, embedding_single, mavs, weibull_models,
                        alpha=2, num_classes=3):
    """
    Apply OpenMax recalibration to a single sample.

    KEY FIX: Revises SOFTMAX PROBABILITIES (not raw logits).
    Uses M1 embedding distances to MAVs (not disease head internals).
    """
    alpha = min(alpha, num_classes)

    # Step 1: Convert logits to softmax probabilities
    exp_logits = np.exp(logits_single - logits_single.max())
    softmax_probs = exp_logits / exp_logits.sum()

    # Step 2: Rank classes by probability (highest first)
    ranked = np.argsort(softmax_probs)[::-1]

    # Step 3: Compute Weibull-based revision for top-alpha classes
    revised = softmax_probs.copy()
    unknown_mass = 0.0

    for rank_idx in range(alpha):
        class_id = ranked[rank_idx]

        # Distance from this sample's embedding to the class MAV
        if class_id in mavs:
            dist = np.linalg.norm(embedding_single - mavs[class_id])
        else:
            dist = 0.0

        # Weibull CDF: probability that a known sample would be THIS far from MAV
        if class_id in weibull_models and dist > 0:
            shape, loc, scale = weibull_models[class_id]
            w_score = stats.weibull_min.cdf(dist, shape, loc=loc, scale=scale)
            w_score = np.clip(w_score, 0.0, 1.0)
        else:
            w_score = 0.0

        # Weight by rank: highest-ranked class gets full weight
        weight = (alpha - rank_idx) / alpha

        # Redistribute probability mass to unknown class
        revision = revised[class_id] * w_score * weight
        revised[class_id] -= revision
        unknown_mass += revision

    # Step 4: Append unknown probability and re-normalize
    openmax_probs = np.append(revised, unknown_mass)

    # Re-normalize to ensure valid probability distribution
    total = openmax_probs.sum()
    if total > 0:
        openmax_probs /= total
    else:
        # Fallback: uniform + unknown
        openmax_probs = np.ones(num_classes + 1) / (num_classes + 1)

    return openmax_probs


def openmax_batch(logits_all, embeddings_all, mavs, weibull_models,
                  alpha=2, num_classes=3):
    """Apply OpenMax to a batch. Returns (openmax_probs, predictions, unknown_probs)."""
    N = len(logits_all)
    openmax_probs = np.zeros((N, num_classes + 1))

    for i in range(N):
        openmax_probs[i] = openmax_recalibrate(
            logits_all[i], embeddings_all[i],
            mavs, weibull_models,
            alpha=alpha, num_classes=num_classes,
        )

    unknown_probs = openmax_probs[:, -1]
    predictions = openmax_probs[:, :-1].argmax(axis=1)

    return openmax_probs, predictions, unknown_probs


print("OpenMax + Weibull implementation loaded (fixed version).")

OpenMax + Weibull implementation loaded (fixed version).


## Cell 13 — Fit OpenMax (MAV + Weibull on Training Data)
Load best disease head, extract training M1 embeddings, compute MAVs, fit Weibull.

In [13]:
# ============================================================
# CELL 13 — FIT OPENMAX ON TRAINING DATA
# ============================================================

# Load best disease head checkpoint
best_path = os.path.join(CFG["ckpt_dir"], "best_model.pth")
if os.path.exists(best_path):
    state = torch.load(best_path, map_location=DEVICE, weights_only=False)
    disease_head.load_state_dict(state["model_state"])
    print(f"Loaded best disease head from epoch {state.get('epoch', '?')}")
else:
    print("WARNING: best_model.pth not found — using current weights")

disease_head.eval()

# ---- Step 1: Extract training features ----
print("\nExtracting training features...")
train_embs, train_acts, train_logits, train_preds, train_labels_arr, train_pids_arr = \
    extract_features(m1_model, disease_head, train_loader, DEVICE)

train_acc = accuracy_score(train_labels_arr, train_preds)
print(f"Training accuracy: {train_acc:.4f}")
print(f"M1 embedding shape: {train_embs.shape}")
print(f"M1 embedding stats: mean={train_embs.mean():.4f}, std={train_embs.std():.4f}")

# ---- Step 2: Compute MAVs on M1 embeddings ----
print("\nComputing MAVs on M1 embeddings (using ALL training samples)...")
mavs, distances = compute_mav_and_distances(
    train_embs, train_labels_arr, CFG["num_known_classes"],
    use_all=True  # KEY: use all samples, not just correctly classified
)

# ---- Step 3: Fit Weibull ----
print(f"\nFitting Weibull (tailsize={CFG['openmax_tailsize']})...")
weibull_models = fit_weibull(distances, tailsize=CFG["openmax_tailsize"])

# ---- Step 4: Diagnostic check ----
# Apply OpenMax to a few training samples and check unknown probabilities
print("\n--- Diagnostic: OpenMax on 10 training samples ---")
for i in range(min(10, len(train_embs))):
    probs = openmax_recalibrate(
        train_logits[i], train_embs[i], mavs, weibull_models,
        alpha=CFG["openmax_alpha"], num_classes=CFG["num_known_classes"]
    )
    unk_p = probs[-1]
    pred = probs[:-1].argmax()
    true_label = train_labels_arr[i]
    print(f"  Sample {i}: true={true_label} pred={pred} unk_prob={unk_p:.4f} "
          f"probs={[f'{p:.3f}' for p in probs]}")

# Save OpenMax parameters
openmax_params = {
    "mavs": {str(k): v.tolist() for k, v in mavs.items()},
    "weibull_models": {str(k): list(v) for k, v in weibull_models.items()},
    "tailsize": CFG["openmax_tailsize"],
    "alpha": CFG["openmax_alpha"],
    "mav_computed_on": "m1_embeddings_256dim",
    "revision_target": "softmax_probabilities",
    "samples_used": "all_training_samples",
}
params_path = os.path.join(CFG["results_dir"], "openmax_params.json")
with open(params_path, "w") as f:
    json.dump(openmax_params, f, indent=2, cls=NumpyEncoder)
print(f"\nSaved OpenMax params to: {params_path}")

Loaded best disease head from epoch 58

Extracting training features...
Training accuracy: 0.9190
M1 embedding shape: (4680, 256)
M1 embedding stats: mean=0.3156, std=0.2507

Computing MAVs on M1 embeddings (using ALL training samples)...
  Class 0 (COPD): 1485 samples, dist mean=3.3254, std=2.4508, min=0.9996, max=27.1333
  Class 1 (Healthy): 1591 samples, dist mean=1.9582, std=0.8413, min=0.8932, max=7.7503
  Class 2 (URTI): 1604 samples, dist mean=3.0482, std=1.9795, min=1.1245, max=10.9858

Fitting Weibull (tailsize=20)...
  Class 0: Weibull fit — shape=5.3121, scale=19.6379 (tailsize=20)
  Class 1: Weibull fit — shape=8.3175, scale=6.9497 (tailsize=20)
  Class 2: Weibull fit — shape=64.1396, scale=10.8709 (tailsize=20)

--- Diagnostic: OpenMax on 10 training samples ---
  Sample 0: true=1 pred=1 unk_prob=0.0017 probs=['0.000', '0.691', '0.307', '0.002']
  Sample 1: true=2 pred=2 unk_prob=0.0000 probs=['0.001', '0.012', '0.987', '0.000']
  Sample 2: true=1 pred=1 unk_prob=0.0138 pr

## Cell 14 — Open-Set Evaluation
Apply OpenMax to:
1. **Known test set** — should get low unknown probability
2. **Unknown set** — should get high unknown probability

Produces AUROC, AUPR, unknown precision/recall.

In [14]:
# ============================================================
# CELL 14 — OPEN-SET EVALUATION
# ============================================================

def compute_classification_metrics(y_true, y_pred, class_names):
    """Standard closed-set classification metrics."""
    acc = accuracy_score(y_true, y_pred)
    prec_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec_macro  = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1_macro   = f1_score(y_true, y_pred, average="macro", zero_division=0)

    prec_per = precision_score(y_true, y_pred, average=None, zero_division=0)
    rec_per  = recall_score(y_true, y_pred, average=None, zero_division=0)
    f1_per   = f1_score(y_true, y_pred, average=None, zero_division=0)

    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)

    per_class = {}
    for i, name in enumerate(class_names):
        support = int((np.array(y_true) == i).sum())
        per_class[name] = {
            "precision": round(float(prec_per[i]), 4) if i < len(prec_per) else 0.0,
            "recall":    round(float(rec_per[i]), 4)  if i < len(rec_per) else 0.0,
            "f1":        round(float(f1_per[i]), 4)   if i < len(f1_per) else 0.0,
            "support":   support,
        }

    return {
        "accuracy": round(float(acc), 4),
        "precision_macro": round(float(prec_macro), 4),
        "recall_macro": round(float(rec_macro), 4),
        "f1_macro": round(float(f1_macro), 4),
        "per_class": per_class,
        "confusion_matrix_raw": cm.tolist(),
        "confusion_matrix_normalized": np.round(cm_norm, 4).tolist(),
    }


def compute_openset_metrics(known_scores, unknown_scores, threshold=0.5):
    """Compute open-set detection metrics (AUROC, AUPR, precision, recall)."""
    y_true = np.concatenate([np.zeros(len(known_scores)), np.ones(len(unknown_scores))])
    y_scores = np.concatenate([known_scores, unknown_scores])

    try:
        auroc = roc_auc_score(y_true, y_scores)
    except ValueError:
        auroc = 0.0
    try:
        aupr = average_precision_score(y_true, y_scores)
    except ValueError:
        aupr = 0.0

    y_pred = (y_scores >= threshold).astype(int)
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))

    unk_prec   = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    unk_recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    return {
        "auroc": round(float(auroc), 4),
        "aupr":  round(float(aupr), 4),
        "unknown_precision": round(float(unk_prec), 4),
        "unknown_recall":    round(float(unk_recall), 4),
        "threshold_used":    threshold,
        "n_known_samples":   len(known_scores),
        "n_unknown_samples": len(unknown_scores),
    }


# ---- Extract test & unknown features ----
print("Extracting known test features...")
test_embs, test_acts, test_logits, test_preds_closed, test_labels, test_pids = \
    extract_features(m1_model, disease_head, test_loader, DEVICE)

print("Extracting unknown features...")
unk_embs, unk_acts, unk_logits, unk_preds_closed, unk_labels, unk_pids = \
    extract_features(m1_model, disease_head, unknown_loader, DEVICE)

# ---- Apply OpenMax ----
print(f"\nApplying OpenMax (alpha={CFG['openmax_alpha']})...")

test_om_probs, test_om_preds, test_unk_probs = openmax_batch(
    test_logits, test_embs, mavs, weibull_models,
    alpha=CFG["openmax_alpha"], num_classes=CFG["num_known_classes"]
)

unk_om_probs, unk_om_preds, unk_unk_probs = openmax_batch(
    unk_logits, unk_embs, mavs, weibull_models,
    alpha=CFG["openmax_alpha"], num_classes=CFG["num_known_classes"]
)

# ---- Distribution statistics (CRITICAL DIAGNOSTIC) ----
print(f"\n{'='*60}")
print("UNKNOWN PROBABILITY DISTRIBUTION (must show separation!)")
print(f"{'='*60}")
print(f"  Known   test: mean={test_unk_probs.mean():.4f}, std={test_unk_probs.std():.4f}, "
      f"median={np.median(test_unk_probs):.4f}, max={test_unk_probs.max():.4f}")
print(f"  Unknown eval: mean={unk_unk_probs.mean():.4f}, std={unk_unk_probs.std():.4f}, "
      f"median={np.median(unk_unk_probs):.4f}, max={unk_unk_probs.max():.4f}")

if test_unk_probs.max() < 0.01 and unk_unk_probs.max() < 0.01:
    print("\n  *** WARNING: All unknown probabilities are near zero! ***")
    print("  *** OpenMax is not redistributing probability mass. ***")
    print("  *** Check MAV distances and Weibull parameters above. ***")

# ---- Closed-set metrics ----
print(f"\n{'='*60}")
print("CLOSED-SET METRICS (known test set)")
print(f"{'='*60}")

closed_metrics = compute_classification_metrics(
    test_labels, test_preds_closed, CFG["known_diseases"]
)
print(f"Accuracy       : {closed_metrics['accuracy']:.4f}")
print(f"Precision(macro): {closed_metrics['precision_macro']:.4f}")
print(f"Recall(macro)  : {closed_metrics['recall_macro']:.4f}")
print(f"F1(macro)      : {closed_metrics['f1_macro']:.4f}")
for cls_name, cls_m in closed_metrics["per_class"].items():
    print(f"  {cls_name:15s}: P={cls_m['precision']:.4f} R={cls_m['recall']:.4f} "
          f"F1={cls_m['f1']:.4f} n={cls_m['support']}")

# ---- Open-set metrics with threshold sweep ----
print(f"\n{'='*60}")
print("OPEN-SET DETECTION METRICS")
print(f"{'='*60}")

thresholds = np.arange(0.05, 0.96, 0.05)
best_os_metrics = None
best_threshold = 0.5
best_harmonic = 0.0

print(f"\n{'Threshold':>10} {'Precision':>10} {'Recall':>10} {'F1':>10} {'AUROC':>10} {'AUPR':>10}")
print("-" * 65)

for thresh in thresholds:
    os_m = compute_openset_metrics(test_unk_probs, unk_unk_probs, threshold=thresh)
    harm = 2 * os_m["unknown_precision"] * os_m["unknown_recall"] / \
           (os_m["unknown_precision"] + os_m["unknown_recall"] + 1e-8)
    print(f"{thresh:>10.2f} {os_m['unknown_precision']:>10.4f} "
          f"{os_m['unknown_recall']:>10.4f} {harm:>10.4f} "
          f"{os_m['auroc']:>10.4f} {os_m['aupr']:>10.4f}")

    if harm > best_harmonic:
        best_harmonic = harm
        best_threshold = thresh
        best_os_metrics = os_m

if best_os_metrics is None:
    best_os_metrics = compute_openset_metrics(test_unk_probs, unk_unk_probs, threshold=0.5)

print(f"\nBest threshold: {best_threshold:.2f}")
print(f"AUROC              : {best_os_metrics['auroc']:.4f}")
print(f"AUPR               : {best_os_metrics['aupr']:.4f}")
print(f"Unknown Precision  : {best_os_metrics['unknown_precision']:.4f}")
print(f"Unknown Recall     : {best_os_metrics['unknown_recall']:.4f}")

Extracting known test features...
Extracting unknown features...

Applying OpenMax (alpha=2)...

UNKNOWN PROBABILITY DISTRIBUTION (must show separation!)
  Known   test: mean=0.0072, std=0.0526, median=0.0000, max=0.9372
  Unknown eval: mean=0.0050, std=0.0369, median=0.0000, max=0.7358

CLOSED-SET METRICS (known test set)
Accuracy       : 0.7689
Precision(macro): 0.4401
Recall(macro)  : 0.5892
F1(macro)      : 0.4561
  COPD           : P=0.9924 R=0.8050 F1=0.8890 n=1467
  Healthy        : P=0.1767 R=0.6667 F1=0.2794 n=66
  URTI           : P=0.1510 R=0.2959 F1=0.2000 n=98

OPEN-SET DETECTION METRICS

 Threshold  Precision     Recall         F1      AUROC       AUPR
-----------------------------------------------------------------
      0.05     0.2745     0.0255     0.0467     0.4516     0.2352
      0.10     0.1842     0.0128     0.0239     0.4516     0.2352
      0.15     0.1600     0.0073     0.0140     0.4516     0.2352
      0.20     0.1053     0.0036     0.0070     0.4516     0.

## Cell 15 — Training Curves & Plots
All required §5 plots plus open-set-specific visualizations.

In [15]:
# ============================================================
# CELL 15 — PLOTS
# ============================================================

results_dir = CFG["results_dir"]
epochs_list = [r["epoch"] for r in history["train"]]

# ---- 1. Loss Curves ----
fig, ax = plt.subplots(figsize=(10, 5), dpi=150)
ax.plot(epochs_list, [r["train_loss"] for r in history["train"]], label="Train", color="#2196F3")
ax.plot(epochs_list, [r["val_loss"] for r in history["train"]], label="Val", color="#FF5722")
if best_epoch >= 0:
    ax.axvline(x=best_epoch, color="gray", ls="--", alpha=0.5, label=f"Best ({best_epoch})")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.set_title("M6 — Disease Head Loss Curves"); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, "loss_curve.png"), dpi=150); plt.show(); plt.close()

# ---- 2. Accuracy Curves ----
fig, ax = plt.subplots(figsize=(10, 5), dpi=150)
ax.plot(epochs_list, [r["train_accuracy"] for r in history["train"]], label="Train", color="#2196F3")
ax.plot(epochs_list, [r["val_accuracy"] for r in history["train"]], label="Val", color="#FF5722")
if best_epoch >= 0:
    ax.axvline(x=best_epoch, color="gray", ls="--", alpha=0.5, label=f"Best ({best_epoch})")
ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")
ax.set_title("M6 — Disease Head Accuracy Curves"); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, "accuracy_curve.png"), dpi=150); plt.show(); plt.close()

# ---- 3. F1 Curves ----
fig, ax = plt.subplots(figsize=(10, 5), dpi=150)
ax.plot(epochs_list, [r["train_f1_macro"] for r in history["train"]], label="Train", color="#2196F3")
ax.plot(epochs_list, [r["val_f1_macro"] for r in history["train"]], label="Val", color="#FF5722")
if best_epoch >= 0:
    ax.axvline(x=best_epoch, color="gray", ls="--", alpha=0.5, label=f"Best ({best_epoch})")
ax.set_xlabel("Epoch"); ax.set_ylabel("Macro F1")
ax.set_title("M6 — Disease Head F1 Curves"); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, "f1_curve.png"), dpi=150); plt.show(); plt.close()

# ---- 4. Confusion Matrix ----
cm = confusion_matrix(test_labels, test_preds_closed)
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CFG["known_diseases"], yticklabels=CFG["known_diseases"], ax=axes[0])
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True"); axes[0].set_title("M6 — CM (Raw)")
sns.heatmap(cm_norm, annot=True, fmt=".3f", cmap="Blues",
            xticklabels=CFG["known_diseases"], yticklabels=CFG["known_diseases"], ax=axes[1])
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True"); axes[1].set_title("M6 — CM (Normalized)")
plt.tight_layout()
plt.savefig(os.path.join(results_dir, "confusion_matrix.png"), dpi=150); plt.show(); plt.close()

# ---- 5. Unknown Probability Distribution ----
fig, ax = plt.subplots(figsize=(10, 5), dpi=150)
ax.hist(test_unk_probs, bins=50, alpha=0.7, label="Known (test)", color="#4CAF50", density=True)
ax.hist(unk_unk_probs, bins=50, alpha=0.7, label="Unknown", color="#F44336", density=True)
ax.axvline(x=best_threshold, color="black", ls="--", lw=1.5,
           label=f"Best threshold ({best_threshold:.2f})")
ax.set_xlabel("OpenMax Unknown Probability"); ax.set_ylabel("Density")
ax.set_title("M6 — Unknown Probability Distribution")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, "unknown_prob_distribution.png"), dpi=150); plt.show(); plt.close()

# ---- 6. ROC Curve ----
y_binary = np.concatenate([np.zeros(len(test_unk_probs)), np.ones(len(unk_unk_probs))])
y_scores = np.concatenate([test_unk_probs, unk_unk_probs])
fpr, tpr, _ = roc_curve(y_binary, y_scores)
fig, ax = plt.subplots(figsize=(7, 7), dpi=150)
ax.plot(fpr, tpr, color="#2196F3", lw=2, label=f"OpenMax (AUROC={best_os_metrics['auroc']:.4f})")
ax.plot([0,1], [0,1], color="gray", ls="--", alpha=0.5, label="Random")
ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title("M6 — Unknown Detection ROC"); ax.legend(loc="lower right"); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, "roc_curve.png"), dpi=150); plt.show(); plt.close()

# ---- 7. PR Curve ----
prec_c, rec_c, _ = precision_recall_curve(y_binary, y_scores)
fig, ax = plt.subplots(figsize=(7, 7), dpi=150)
ax.plot(rec_c, prec_c, color="#FF9800", lw=2, label=f"OpenMax (AUPR={best_os_metrics['aupr']:.4f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("M6 — Unknown Detection PR Curve"); ax.legend(loc="lower left"); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, "pr_curve.png"), dpi=150); plt.show(); plt.close()

print(f"All plots saved to: {results_dir}")

All plots saved to: /kaggle/working/results_M6


## Cell 16 — Results JSON (Protocol-Compliant)
Generates `results_M6.json` per Model Training Protocol §4.

In [16]:
# ============================================================
# CELL 16 — RESULTS JSON
# ============================================================

# Inference time measurement
disease_head.eval()
n_inference = min(100, len(test_ds))
inference_times = []
for i in range(n_inference):
    spec, _, _ = test_ds[i]
    spec = spec.unsqueeze(0).to(DEVICE)
    start_t = time.time()
    with torch.no_grad():
        emb = m1_model.get_embedding(spec)
        _, _ = disease_head(emb)
    inference_times.append((time.time() - start_t) * 1000)
inference_ms = np.mean(inference_times)
print(f"Inference time: {inference_ms:.2f} ms/sample (over {n_inference} samples)")

# Model size
head_size_mb = get_model_size_mb(disease_head)
trainable_params = sum(p.numel() for p in disease_head.parameters() if p.requires_grad)
total_params_head = sum(p.numel() for p in disease_head.parameters())
total_params_m1 = sum(p.numel() for p in m1_model.parameters())
total_params_all = total_params_m1 + total_params_head
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

results = {
    "meta": {
        "model_id": "M6",
        "model_name": "OpenMax + Weibull Baseline",
        "member": "B",
        "member_name": "Member B (Disease Diagnosis & OWL)",
        "date_completed": datetime.datetime.now().strftime("%Y-%m-%d"),
        "is_augmented": False,
        "augmentation_method": "none",
        "notes": "Rebuilt from scratch. MAV on M1 256-dim embeddings, OpenMax revises softmax probs, all training samples used for MAV."
    },

    "config": {
        "sample_rate": CFG["sample_rate"],
        "n_mels": CFG["n_mels"],
        "n_fft": CFG["n_fft"],
        "hop_length": CFG["hop_length"],
        "win_length": CFG["win_length"],
        "f_min": CFG["f_min"],
        "f_max": CFG["f_max"],
        "batch_size": CFG["batch_size"],
        "num_epochs": CFG["num_epochs"],
        "lr": CFG["lr"],
        "optimizer": "Adam",
        "scheduler": "StepLR",
        "architecture": "M1_CNN_frozen + DiseaseHead(256->128->3)",
        "seed": CFG["seed"],
        "openmax_tailsize": CFG["openmax_tailsize"],
        "openmax_alpha": CFG["openmax_alpha"],
        "openmax_threshold": best_threshold,
    },

    "environment": {
        "platform": "Kaggle",
        "gpu_name": gpu_name,
        "pytorch_version": torch.__version__,
        "python_version": f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}",
    },

    "dataset_info": {
        "dataset": "ICBHI_2017",
        "task": "disease_diagnosis_open_set",
        "known_classes": CFG["known_diseases"],
        "unknown_classes": CFG["unknown_diseases"],
        "train_samples": len(df_known_train),
        "test_samples_known": len(df_known_test),
        "test_samples_unknown": len(df_unknown),
        "split_method": "patient_independent_70_30",
        "n_train_patients": int(df_known_train["patient_id"].nunique()),
        "n_test_patients_known": int(df_known_test["patient_id"].nunique()),
        "n_test_patients_unknown": int(df_unknown["patient_id"].nunique()),
    },

    "efficiency": {
        "total_params": int(total_params_all),
        "trainable_params": int(trainable_params),
        "total_params_head_only": int(total_params_head),
        "model_size_mb": head_size_mb,
        "training_time_total_s": round(total_training_time, 1),
        "training_time_per_epoch_s_avg": round(avg_epoch_time, 1),
        "gpu_name": gpu_name,
        "inference_time_ms_per_sample": round(inference_ms, 2),
    },

    "best_epoch": {
        "epoch": best_epoch,
        "primary_metric": "val_f1_macro",
        "primary_metric_value": round(best_f1, 4),
    },

    "best_metrics": {
        **closed_metrics,
        "open_set": best_os_metrics,
    },

    "ablation": {
        "ablation_group": "rejection_method",
        "ablation_role": "variant",
        "baseline_model_id": "M15",
        "variable_changed": "rejection_method: OpenMax_Weibull",
        "variables_held_constant": [
            "backbone: M1_CNN_frozen",
            "data_split: patient_independent_70_30",
            "known_classes: COPD_Healthy_URTI",
            "unknown_classes: Bronchiectasis_Pneumonia_Bronchiolitis",
            "augmentation: none",
            "seed: 42",
        ],
        "component_flags": {
            "has_sound_event_head": False,
            "has_disease_head": True,
            "has_cross_task_consistency": False,
            "has_cqkd_regularization": False,
            "has_openmax_rejection": True,
            "owl_stage": 0,
            "compression_clusters": None,
        },
        "loss_weights": {
            "sound_event_weight": None,
            "disease_weight": 1.0,
            "consistency_weight": None,
        }
    },

    "training_history": history["train"],
}

results_path = os.path.join(results_dir, "results_M6.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2, cls=NumpyEncoder)
print(f"Saved: {results_path}")

Inference time: 2.52 ms/sample (over 100 samples)
Saved: /kaggle/working/results_M6/results_M6.json


## Cell 17 — Patient-Level Aggregation
Disease is a patient-level attribute. Aggregate cycle-level OpenMax scores per patient.

In [17]:
# ============================================================
# CELL 17 — PATIENT-LEVEL AGGREGATION
# ============================================================

def patient_level_analysis(unk_probs, patient_ids, threshold, is_known=True):
    """Aggregate cycle-level unknown probabilities to patient level."""
    df_temp = pd.DataFrame({"patient_id": patient_ids, "unk_prob": unk_probs})
    agg = df_temp.groupby("patient_id")["unk_prob"].agg(["mean", "max", "count"]).reset_index()
    agg.columns = ["patient_id", "mean_unk_prob", "max_unk_prob", "n_cycles"]

    agg["flagged_mean"] = agg["mean_unk_prob"] >= threshold
    agg["flagged_max"]  = agg["max_unk_prob"]  >= threshold

    group = "KNOWN" if is_known else "UNKNOWN"
    print(f"\n{'='*65}")
    print(f"PATIENT-LEVEL ANALYSIS — {group} ({len(agg)} patients)")
    print(f"{'='*65}")
    print(f"{'Patient':>10} {'Cycles':>7} {'Mean UnkP':>10} {'Max UnkP':>10} {'Flag(mean)':>11} {'Flag(max)':>10}")
    print("-" * 65)

    for _, row in agg.sort_values("mean_unk_prob", ascending=False).iterrows():
        fm = "YES" if row["flagged_mean"] else "no"
        fx = "YES" if row["flagged_max"]  else "no"
        print(f"{int(row['patient_id']):>10} {int(row['n_cycles']):>7} "
              f"{row['mean_unk_prob']:>10.4f} {row['max_unk_prob']:>10.4f} "
              f"{fm:>11} {fx:>10}")

    n_fm = agg["flagged_mean"].sum()
    n_fx = agg["flagged_max"].sum()
    total = len(agg)

    print(f"\nFlagged (mean thresh): {n_fm}/{total}")
    print(f"Flagged (max thresh) : {n_fx}/{total}")

    if is_known:
        print(f"False positive rate (mean): {n_fm/total:.4f}")
        print(f"False positive rate (max) : {n_fx/total:.4f}")
    else:
        print(f"Detection rate (mean): {n_fm/total:.4f}")
        print(f"Detection rate (max) : {n_fx/total:.4f}")

    return agg


known_agg   = patient_level_analysis(test_unk_probs, test_pids, best_threshold, is_known=True)
unknown_agg = patient_level_analysis(unk_unk_probs, unk_pids, best_threshold, is_known=False)


PATIENT-LEVEL ANALYSIS — KNOWN (32 patients)
   Patient  Cycles  Mean UnkP   Max UnkP  Flag(mean)  Flag(max)
-----------------------------------------------------------------
       118      40     0.0635     0.7979         YES        YES
       150      17     0.0561     0.9372         YES        YES
       193     138     0.0472     0.5428          no        YES
       139      46     0.0099     0.1802          no        YES
       187      17     0.0062     0.0667          no        YES
       154     208     0.0039     0.2148          no        YES
       114      35     0.0035     0.0482          no         no
       113      50     0.0015     0.0159          no         no
       192      46     0.0014     0.0253          no         no
       210      31     0.0003     0.0057          no         no
       179      15     0.0003     0.0019          no         no
       180      30     0.0003     0.0024          no         no
       133     156     0.0002     0.0161          no    

## Cell 18 — OpenMax Hyperparameter Sensitivity
Sweep tailsize × alpha for the paper's appendix.

In [18]:
# ============================================================
# CELL 18 — SENSITIVITY ANALYSIS
# ============================================================

tailsizes = [5, 10, 15, 20, 30, 50]
alphas    = [1, 2, 3]

sensitivity_results = []

print(f"{'Tailsize':>10} {'Alpha':>6} {'AUROC':>8} {'AUPR':>8} {'BestThr':>8} {'Prec':>8} {'Recall':>8}")
print("-" * 70)

for ts in tailsizes:
    wb = fit_weibull(distances, tailsize=ts)

    for a in alphas:
        _, _, t_unk_p = openmax_batch(
            test_logits, test_embs, mavs, wb,
            alpha=a, num_classes=CFG["num_known_classes"]
        )
        _, _, u_unk_p = openmax_batch(
            unk_logits, unk_embs, mavs, wb,
            alpha=a, num_classes=CFG["num_known_classes"]
        )

        best_h = 0
        best_t = 0.5
        best_m = None
        for t in np.arange(0.05, 0.95, 0.05):
            m = compute_openset_metrics(t_unk_p, u_unk_p, threshold=t)
            h = 2 * m["unknown_precision"] * m["unknown_recall"] / \
                (m["unknown_precision"] + m["unknown_recall"] + 1e-8)
            if h > best_h:
                best_h = h
                best_t = t
                best_m = m

        if best_m is None:
            best_m = compute_openset_metrics(t_unk_p, u_unk_p, threshold=0.5)

        print(f"{ts:>10} {a:>6} {best_m['auroc']:>8.4f} {best_m['aupr']:>8.4f} "
              f"{best_t:>8.2f} {best_m['unknown_precision']:>8.4f} {best_m['unknown_recall']:>8.4f}")

        sensitivity_results.append({
            "tailsize": ts, "alpha": a,
            "auroc": best_m["auroc"], "aupr": best_m["aupr"],
            "best_threshold": best_t,
            "precision": best_m["unknown_precision"],
            "recall": best_m["unknown_recall"],
        })

sens_path = os.path.join(CFG["results_dir"], "openmax_sensitivity.json")
with open(sens_path, "w") as f:
    json.dump(sensitivity_results, f, indent=2, cls=NumpyEncoder)
print(f"\nSaved: {sens_path}")

  Tailsize  Alpha    AUROC     AUPR  BestThr     Prec   Recall
----------------------------------------------------------------------
  Class 0: Weibull fit — shape=7.2412, scale=22.0911 (tailsize=10)
  Class 1: Weibull fit — shape=14.7089, scale=7.5065 (tailsize=10)
  Class 2: Weibull fit — shape=297.9139, scale=10.9819 (tailsize=10)
         5      1   0.3851   0.2076     0.05   0.1000   0.0036
         5      2   0.4077   0.2168     0.05   0.1429   0.0055
         5      3   0.4233   0.2228     0.05   0.1429   0.0055
  Class 0: Weibull fit — shape=7.2412, scale=22.0911 (tailsize=10)
  Class 1: Weibull fit — shape=14.7089, scale=7.5065 (tailsize=10)
  Class 2: Weibull fit — shape=297.9139, scale=10.9819 (tailsize=10)
        10      1   0.3851   0.2076     0.05   0.1000   0.0036
        10      2   0.4077   0.2168     0.05   0.1429   0.0055
        10      3   0.4233   0.2228     0.05   0.1429   0.0055
  Class 0: Weibull fit — shape=6.0717, scale=20.7327 (tailsize=15)
  Class 1: Weib

## Cell 19 — Done Checklist
Verify all protocol requirements before calling M6 complete.

In [19]:
# ============================================================
# CELL 19 — DONE CHECKLIST
# ============================================================

print("=" * 60)
print("M6 — DONE CHECKLIST")
print("=" * 60)

checks = [
    ("Results JSON",              os.path.exists(os.path.join(results_dir, "results_M6.json"))),
    ("Best model checkpoint",     os.path.exists(os.path.join(CFG["ckpt_dir"], "best_model.pth"))),
    ("Loss curve",                os.path.exists(os.path.join(results_dir, "loss_curve.png"))),
    ("Accuracy curve",            os.path.exists(os.path.join(results_dir, "accuracy_curve.png"))),
    ("F1 curve",                  os.path.exists(os.path.join(results_dir, "f1_curve.png"))),
    ("Confusion matrix",          os.path.exists(os.path.join(results_dir, "confusion_matrix.png"))),
    ("Unknown prob distribution", os.path.exists(os.path.join(results_dir, "unknown_prob_distribution.png"))),
    ("ROC curve",                 os.path.exists(os.path.join(results_dir, "roc_curve.png"))),
    ("PR curve",                  os.path.exists(os.path.join(results_dir, "pr_curve.png"))),
    ("OpenMax params",            os.path.exists(os.path.join(results_dir, "openmax_params.json"))),
    ("Sensitivity analysis",      os.path.exists(os.path.join(results_dir, "openmax_sensitivity.json"))),
    ("Patient-independent split", True),
    ("Augmentation = none",       True),
    ("Ablation block filled",     True),
    ("Inference time measured",   inference_ms > 0),
    ("AUROC > 0 (sanity)",        best_os_metrics["auroc"] > 0.0),
]

all_pass = True
for name, ok in checks:
    status = "PASS" if ok else "FAIL"
    if not ok: all_pass = False
    print(f"  [{status}] {name}")

print(f"\n{'='*60}")
print("ALL CHECKS PASSED" if all_pass else "SOME CHECKS FAILED")
print(f"{'='*60}")

print(f"\nKey Results:")
print(f"  Closed-set F1 (macro)    : {best_f1:.4f}")
print(f"  Closed-set Accuracy      : {closed_metrics['accuracy']:.4f}")
print(f"  Open-set AUROC           : {best_os_metrics['auroc']:.4f}")
print(f"  Open-set AUPR            : {best_os_metrics['aupr']:.4f}")
print(f"  Unknown Detect Precision : {best_os_metrics['unknown_precision']:.4f}")
print(f"  Unknown Detect Recall    : {best_os_metrics['unknown_recall']:.4f}")
print(f"  Trainable params (head)  : {trainable_params:,}")
print(f"  Total params (M1+head)   : {total_params_all:,}")
print(f"  Head size                : {head_size_mb} MB")
print(f"  Inference time           : {inference_ms:.2f} ms/sample")
print(f"  Best threshold           : {best_threshold:.2f}")
print(f"\nThis model's AUROC/AUPR are the comparison target for M15.")
print(f"M15 (cross-task consistency) must beat these numbers.")

M6 — DONE CHECKLIST
  [PASS] Results JSON
  [PASS] Best model checkpoint
  [PASS] Loss curve
  [PASS] Accuracy curve
  [PASS] F1 curve
  [PASS] Confusion matrix
  [PASS] Unknown prob distribution
  [PASS] ROC curve
  [PASS] PR curve
  [PASS] OpenMax params
  [PASS] Sensitivity analysis
  [PASS] Patient-independent split
  [PASS] Augmentation = none
  [PASS] Ablation block filled
  [PASS] Inference time measured
  [PASS] AUROC > 0 (sanity)

ALL CHECKS PASSED

Key Results:
  Closed-set F1 (macro)    : 0.4561
  Closed-set Accuracy      : 0.7689
  Open-set AUROC           : 0.4516
  Open-set AUPR            : 0.2352
  Unknown Detect Precision : 0.2745
  Unknown Detect Recall    : 0.0255
  Trainable params (head)  : 33,283
  Total params (M1+head)   : 455,015
  Head size                : 0.13 MB
  Inference time           : 2.52 ms/sample
  Best threshold           : 0.05

This model's AUROC/AUPR are the comparison target for M15.
M15 (cross-task consistency) must beat these numbers.


---
## Notes for M15 Comparison

**How M15 should compare against this baseline:**

```python
import json
with open('/kaggle/input/<m6-output>/results_M6.json') as f:
    m6 = json.load(f)

m6_auroc = m6['best_metrics']['open_set']['auroc']
m6_aupr  = m6['best_metrics']['open_set']['aupr']

print(f"M6 AUROC: {m6_auroc} — M15 must exceed this")
print(f"M6 AUPR:  {m6_aupr}  — M15 must exceed this")
```

The comparison table is the paper's headline result.